# 02 · Difficulty classification | Riwaq AI

**Purpose.** Classify the reading or learning level of English educational text into `EASY`, `INTERMEDIATE`, or `ADVANCED` for content discovery.

**Reading guide.** Curated record of one executed Kaggle training run. Code excerpts are selected from `notebook48347d88fe.ipynb`, cell 1; metrics are transcribed from its saved outputs. The dataset CSVs and GPU environment are external, so this is an experiment record rather than a self-contained rerun.

## 1. Frame the problem and validate input

Difficulty is **single-label**: each example has exactly one of three levels, so the classifier uses a softmax distribution and `argmax`. I checked that the `text` and `difficulty` columns existed, filled missing text, standardized label capitalization, and rejected unknown labels before training. The input was supplied as three separate Kaggle CSV files; I did not regenerate the split inside the notebook.

In [ ]:
# Selected from the original training cell; requires the Kaggle dataset attached.
from pathlib import Path
import pandas as pd

LABELS = ["EASY", "INTERMEDIATE", "ADVANCED"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

def find_input(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        raise FileNotFoundError(filename)
    return matches[0]

train_df = pd.read_csv(find_input("binx_difficulty_train.csv"))
val_df = pd.read_csv(find_input("binx_difficulty_validation.csv"))
test_df = pd.read_csv(find_input("binx_difficulty_test.csv"))
for frame in (train_df, val_df, test_df):
    assert {"text", "difficulty"}.issubset(frame.columns)
    frame["text"] = frame["text"].fillna("").astype(str)
    frame["difficulty"] = frame["difficulty"].astype(str).str.upper()
    assert set(frame["difficulty"]).issubset(LABEL2ID)
    frame["label_id"] = frame["difficulty"].map(LABEL2ID)

**Recorded input:** 33,941 train · 7,274 validation · 7,273 test examples. Training classes were balanced: 11,350 EASY, 11,336 INTERMEDIATE, 11,255 ADVANCED. The source notebook does not document how these external CSV labels were collected or audited, so results should be interpreted with that provenance limit.

## 2. Choose a compact transformer and a measurable objective

I fine-tuned `distilroberta-base` with a 256-token maximum. Dynamic padding avoided padding every example to 256 tokens. The training run used **two epochs**, learning rate `2e-5`, batch sizes 16/32 (train/evaluation), weight decay `0.01`, and selection of the best validation checkpoint by **Macro-F1**. Macro-F1 makes performance on each difficulty level visible even if the class mix changes.

In [ ]:
# Configuration and metric logic from the executed Kaggle cell.
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
)
model.config.problem_type = "single_label_classification"

def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    actual = eval_prediction.label_ids
    return {
        "accuracy": accuracy_score(actual, predicted),
        "f1_macro": f1_score(actual, predicted, average="macro", zero_division=0),
        "f1_weighted": f1_score(actual, predicted, average="weighted", zero_division=0),
    }

In [ ]:
# Key training settings in the executed notebook (Trainer creation omitted).
training_settings = {
    "num_train_epochs": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "seed": 42,
}
# The original used an epoch evaluation/save strategy and handled Transformers
# versions with either `eval_strategy` or `evaluation_strategy`.

## 3. Validate, then use the untouched test split

| Measure | Validation | Held-out test |
|---|---:|---:|
| Accuracy | 0.9715 | 0.9702 |
| Macro-F1 | 0.9714 | 0.9701 |
| Weighted-F1 | 0.9715 | 0.9701 |

The validation checkpoint was chosen before the final test evaluation. The close validation/test values are reassuring for this split, but they do not establish how the model generalizes to future creator posts from a different source.

| Test class | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
| EASY | 0.9593 | 0.9869 | 0.9729 | 2,367 |
| INTERMEDIATE | 0.9681 | 0.9466 | 0.9572 | 2,434 |
| ADVANCED | 0.9829 | 0.9773 | 0.9801 | 2,472 |

`INTERMEDIATE` was the hardest level here, with the lowest recall and F1. A useful next check would review real intermediate posts and adjacent-level confusion, rather than treating the overall 97% accuracy as sufficient. **Source:** executed notebook, cell 1 output.

## 4. Save and integrate

The run saved the model, tokenizer, `label_map.json`, a model report, CSV test predictions and a downloadable ZIP. The deployed service reads the three label names from `../app/models/difficulty_model/label_map.json` and uses a **softmax** score to choose one class. A high test result on the supplied CSVs does not validate the difficulty definitions themselves; that requires review of label construction and representative product posts.

**Trace:** `notebook48347d88fe.ipynb`, cell 1 and its output; `../app/models/difficulty_model/`; `../app/services/model_service.py`.